In [0]:
#path = /Volumes/workspace/default/pyspark/Sample - Superstore.csv

In [0]:
from pyspark.sql import SparkSession
from  pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
superstore_schema = StructType([
  StructField("Row ID", IntegerType(), True),
  StructField("Order ID", StringType(), True),
  StructField("Order Date", StringType(), True),
  StructField("Ship Date", StringType(), True),
  StructField("Ship Mode", StringType(), True),
  StructField("Customer ID", StringType(), True),
  StructField("Customer Name", StringType(), True),
  StructField("Segment", StringType(), True),
  StructField("Country", StringType(), True),
  StructField("City", StringType(), True),
  StructField("State", StringType(), True),
  StructField("Postal Code", StringType(), True),
  StructField("Region", StringType(), True),
  StructField("Product ID", StringType(), True),
  StructField("Category", StringType(), True),
  StructField("Sub-Category", StringType(), True),
  StructField("Product Name", StringType(), True),
  StructField("Sales", DoubleType(), True),
  StructField("Quantity", IntegerType(), True),
  StructField("Discount", DoubleType(), True),
  StructField("Profit", DoubleType(), True)
])

In [0]:
df = spark.read.format ("csv") \
    .option ("header", "true") \
    .schema(superstore_schema) \
    .load("/Volumes/workspace/default/pyspark/Sample - Superstore.csv")


In [0]:
df = df.withColumn(
        "Order Date", expr("try_to_date(`Order Date`, 'MM/dd/yyyy')")
    ).withColumn(
        "Ship Date", expr("try_to_date(`Ship Date`, 'MM/dd/yyyy')")
    )

In [0]:
from pyspark.sql.window import Window

In [0]:
window_spec = Window.partitionBy ("Category").orderBy(col("Sales").desc())

In [0]:
df_row = df.withColumn("row_num", row_number().over (window_spec))

df_row.filter (col("row_num") ==1)

DataFrame[Row ID: int, Order ID: string, Order Date: date, Ship Date: date, Ship Mode: string, Customer ID: string, Customer Name: string, Segment: string, Country: string, City: string, State: string, Postal Code: string, Region: string, Product ID: string, Category: string, Sub-Category: string, Product Name: string, Sales: double, Quantity: int, Discount: double, Profit: double, row_num: int]

In [0]:
# Top selling product in each category including ties

df_rank = df.withColumn("rank_number", rank().over(window_spec))
df_filterd1 = df_rank.filter(col("rank_number") <= 3)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4853038395507676>, line 3
      1 # Top selling product in each category including ties
----> 3 df_rank = df.withColumn("rank_number", rank().over(window_spec))
      4 df_filterd1 = df_rank.filter(col("rank_number") <= 3)

NameError: name 'df' is not defined

In [0]:
# lag = previous
#lead = next

from pyspark.sql.functions import lag, lead

# Analyze customer behavior over time

window_spec = Window.partationBy ("Customer ID)").overBy("Order Date")

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-4853038395507677>, line 8
      4 from pyspark.sql.functions import lag, lead
      6 # Analyze customer behavior over time
----> 8 window_spec = Window.partationBy ("Customer ID)").overBy("Order Date")

AttributeError: type object 'Window' has no attribute 'partationBy'

In [0]:
df_lag = df.withColumn (
    "previous_sales",
    lag("Sales", 1).over(window_spec)
)


df_lag.select("Customer ID", "Order Date", "Sales", "previous_sales").show()

+-----------+----------+--------+--------------+
|Customer ID|Order Date|   Sales|previous_sales|
+-----------+----------+--------+--------------+
|   TP-21415|      NULL|4416.174|          NULL|
|   QJ-19255|      NULL|  4404.9|      4416.174|
|   JH-15985|      NULL|4297.644|        4404.9|
|   PF-19120|      NULL|4228.704|      4297.644|
|   GT-14710|2014-11-17| 4007.84|      4228.704|
|   LW-17215|      NULL|3785.292|       4007.84|
|   NP-18700|2014-12-12|3610.848|      3785.292|
|   LA-16780|      NULL|  3504.9|      3610.848|
|   MS-17365|2016-11-26|3406.664|        3504.9|
|   EB-13840|      NULL| 3393.68|      3406.664|
|   TB-21520|      NULL| 3083.43|       3393.68|
|   SV-20365|      NULL|2888.127|       3083.43|
|   SC-20230|      NULL|2887.056|      2888.127|
|   RP-19390|      NULL| 2807.84|      2887.056|
|   CJ-12010|      NULL| 2803.92|       2807.84|
|   SC-20305|      NULL| 2803.92|       2803.92|
|   AS-10285|2016-11-11| 2678.94|       2803.92|
|   LC-16960|      N

In [0]:
# find next order sale for each customer
df_lead = df.withColumn (
    "next_sales",
    lead("Sales", 1).over(window_spec)
)


df_lead.select("Customer ID", "Order Date", "Sales", "next_sales").show()

+-----------+----------+--------+----------+
|Customer ID|Order Date|   Sales|next_sales|
+-----------+----------+--------+----------+
|   TP-21415|      NULL|4416.174|    4404.9|
|   QJ-19255|      NULL|  4404.9|  4297.644|
|   JH-15985|      NULL|4297.644|  4228.704|
|   PF-19120|      NULL|4228.704|   4007.84|
|   GT-14710|2014-11-17| 4007.84|  3785.292|
|   LW-17215|      NULL|3785.292|  3610.848|
|   NP-18700|2014-12-12|3610.848|    3504.9|
|   LA-16780|      NULL|  3504.9|  3406.664|
|   MS-17365|2016-11-26|3406.664|   3393.68|
|   EB-13840|      NULL| 3393.68|   3083.43|
|   TB-21520|      NULL| 3083.43|  2888.127|
|   SV-20365|      NULL|2888.127|  2887.056|
|   SC-20230|      NULL|2887.056|   2807.84|
|   RP-19390|      NULL| 2807.84|   2803.92|
|   CJ-12010|      NULL| 2803.92|   2803.92|
|   SC-20305|      NULL| 2803.92|   2678.94|
|   AS-10285|2016-11-11| 2678.94|  2676.672|
|   LC-16960|      NULL|2676.672|   2665.62|
|   SF-20200|      NULL| 2665.62|  2621.322|
|   ED-138